In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import os
class CNNModel:
    def __init__(self, input_shape=(128, 128, 3), num_classes=2):
        self.input_shape = input_shape
        self.num_classes = num_classes
        self.model = self.build_model()
        self.model.compile(
            loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
            optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
            metrics=["accuracy"],
        )

    def _MBConvBlock(
        self, x, out_channels, kernel_size=3, use_se=False, strides=1, r=4, t=6
    ):
        # -----expand(1*1)--------
        in_channels = x.shape[-1]
        shortcut = x
        Z1 = layers.Conv2D(in_channels * t, 1, padding="same", use_bias=False)(x)
        Z1 = layers.BatchNormalization()(Z1)
        Z1 = layers.Activation("swish")(Z1)
        # ---------depthwise( k*k) mỗi channel có 1 kernel riêng (k × k × 1)
        Z1 = layers.DepthwiseConv2D(
            kernel_size=kernel_size, strides=strides, padding="same", use_bias=False
        )(Z1)
        Z1 = layers.BatchNormalization()(Z1)
        Z1 = layers.Activation("swish")(Z1)
        if use_se:  # Squeeze-and-Excitation(optional)-tự động học “channel nào quan trọng hơn” trong feature map.
            C = Z1.shape[-1]  # số channel thực tế

            # ---------Squeeze-------
            u = layers.GlobalAveragePooling2D()(Z1)  # u.shape = (None, C)
            # ---------Excitation-------
            v1 = layers.Dense(C // r, activation="swish", use_bias=False)(
                u
            )  # (batch, C // r)
            v2 = layers.Dense(C, activation="sigmoid", use_bias=False)(v1)
            v2 = layers.Reshape((1, 1, C))(
                v2
            )  # reshape về (None, 1, 1, C) để phù hợp với đầu vào của Multiply layer
            # ---------------Scale------
            Z1 = layers.Multiply()([Z1, v2])
        Y = layers.Conv2D(
            filters=out_channels,
            kernel_size=1,
            padding="same",
            use_bias=False,
        )(Z1)
        Y = layers.BatchNormalization()(Y)
        # ---------skip connection----------
        if (strides == 1) and (in_channels == out_channels):
            return layers.Add()([Y, shortcut])
        return Y

    def _Fused_MBConvBlock(
        self, x, out_channels, kernel_size=3, strides=1, use_se=False, r=4, t=6
    ):
        shortcut = x
        in_channels = x.shape[-1]
        Z1 = layers.Conv2D(
            filters=in_channels * t,
            kernel_size=kernel_size,
            strides=strides,
            padding="same",
            use_bias=False,
        )(
            x
        )  # x.shape = (batch, H, W, C_in)-> Z1.shape = (batch, H/strides, W/strides, out_channels)
        Z1 = layers.BatchNormalization()(Z1)
        Z1 = layers.Activation("swish")(Z1)

        if use_se:  # Squeeze-and-Excitation(optional)-tự động học “channel nào quan trọng hơn” trong feature map.
            C = Z1.shape[-1]  # số channel thực tế
            # ---------Squeeze---------
            u = layers.GlobalAveragePooling2D()(Z1)  # u.shape = (None, C)
            # ---------Excitation---------
            v1 = layers.Dense(C // r, activation="swish", use_bias=False)(
                u
            )  # (batch, C // r)
            v2 = layers.Dense(C, activation="sigmoid", use_bias=False)(v1)
            v2 = layers.Reshape((1, 1, C))(
                v2
            )  # reshape về (None, 1, 1, C) để phù hợp với đầu vào của Multiply layer
            # ---------------Scale----------
            Z1 = layers.Multiply()([Z1, v2])

        # ------------- project 1x1-----------
        Y = layers.Conv2D(
            filters=out_channels,
            kernel_size=1,
            padding="same",
            use_bias=False,
        )(Z1)
        Y = layers.BatchNormalization()(Y)
        # -------------skip connection-------------
        if (strides == 1) and (in_channels == out_channels):
            return layers.Add()([Y, x])
        return Y

    def build_model(self):
        _input = tf.keras.Input(shape=self.input_shape)
        x = layers.RandomFlip("horizontal")(_input)
        x = layers.RandomRotation(0.2)(x)
        x = layers.RandomZoom(0.2)(x)
        # ----Stage 0: Conv3x3, Stride 2, 24 channels------
        x = layers.Conv2D(24, (3, 3), activation="relu", padding="same", strides=2)(x)
        x = layers.BatchNormalization()(x)

        # ------Stage 1: Fused-MBConv1, 2 layers, Stride 1, 24 channels------
        for _ in range(2):
            x = self._Fused_MBConvBlock(
                x, out_channels=24, kernel_size=3, strides=1, use_se=False, r=4, t=1
            )
        # -------Stage 2: Fused-MBConv4, 4 layers, Stride 2, 48 channels------
        for i in range(4):
            stride = 2 if i == 0 else 1  # Chỉ giảm ảnh ở layer đầu tiên của stage
            x = self._Fused_MBConvBlock(x, out_channels=48, strides=stride, t=4)
        # -------Stage 3: Fused-MBConv4, 4 layers, Stride 2, 64 channels------
        for i in range(4):
            stride = 2 if i == 0 else 1
            x = self._Fused_MBConvBlock(x, out_channels=64, strides=stride, t=4)
        # -------Stage 4: MBConv4, 6 layers, Stride 2, 128 channels------
        for i in range(6):
            stride = 2 if i == 0 else 1
            x = self._MBConvBlock(x, out_channels=128, strides=stride, use_se=True, t=4)
        ## Stage 5: MBConv6, 9 layers, Stride 1, 160 channels, SE
        for _ in range(9):
            x = self._Fused_MBConvBlock(
                x, out_channels=160, kernel_size=3, strides=1, use_se=False, r=4, t=6
            )
        # Stage 6: MBConv6, 15 layers, Stride 2, 256 channels, SE
        for i in range(15):
            stride = 2 if i == 0 else 1
            x = self._MBConvBlock(x, out_channels=256, strides=stride, use_se=True, t=6)
        x = layers.Conv2D(1280, (1, 1), activation="relu", padding="same")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("swish")(x)

        x = layers.GlobalMaxPooling2D()(x)
        x = layers.Flatten()(x)

        x = layers.Dense(128, activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.3)(x)
        outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)
        model = tf.keras.Model(inputs=_input, outputs=outputs)
        return model

    def train(self, data, epochs=50, batch_size=16):
        # Assuming data is a dictionary with keys 'X_train', 'y_train', 'X_val', 'y_val'
        X_train = data["X_train"]
        y_train = data["y_train"]
        X_val = data["X_val"]
        y_val = data["y_val"]

        lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=5, min_lr=1e-5, verbose=1
        )
        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=12, restore_best_weights=True, verbose=1
        )
        return self.model.fit(
            X_train,
            y_train,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=[lr_scheduler, early_stop],
        )

    def evaluate(self, X_test, y_test):
        return self.model.evaluate(X_test, y_test)

    def predict(self, img):
        """
        img: Một ảnh duy nhất đã được load bằng cv2 và resize về (128, 128)
        """
        img_batch = np.expand_dims(img, axis=0)
        prediction = self.model.predict(img_batch, verbose=0)
        probability = prediction[0][0]
        class_id = 1 if probability > 0.5 else 0
        return class_id, probability

    def save(self, file_path="models/face_verify_v1.keras"):
        os.makedirs(os.path.dirname(file_path), exist_ok=True)
        self.model.save(file_path)
        print(f"--- Đã lưu model thành công tại: {file_path} ---")


In [ ]:
folder_path_drive="data/data"
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
!cp -r /content/drive/MyDrive/data/data.zip .

In [ ]:
%cd /content
!unzip -q data.zip -d /content/data

In [ ]:
import os
import cv2
import numpy as np

def load_yolo_style_data(root_path, subset):
    """
    subset: 'train', 'val', hoặc 'test'
    """
    images_dir = os.path.join(root_path, subset, "images")
    labels_dir = os.path.join(root_path, subset, "labels")

    X, y = [], []

    # Lấy danh sách ảnh
    image_files = [f for f in os.listdir(images_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]

    for img_name in image_files:
        # 1. Đọc ảnh
        img_path = os.path.join(images_dir, img_name)
        img = cv2.imread(img_path)
        if img is None: continue
        img = cv2.resize(img, (128, 128))

        # 2. Đọc nhãn tương ứng (đổi đuôi ảnh thành .txt)
        label_name = os.path.splitext(img_name)[0] + ".txt"
        label_path = os.path.join(labels_dir, label_name)

        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                first_line = f.readline().strip()
                if first_line:
                    class_id = int(first_line.split()[0]) # Lấy số đầu tiên là class
                    X.append(img)
                    y.append(class_id)

    return np.array(X) / 255.0, np.array(y).reshape(-1, 1)

def main():
    model_wrapper = CNNModel()

    print("Đang bốc dữ liệu từ folder images & labels...")
    X_train, y_train = load_yolo_style_data("/content/data", "train")
    X_val, y_val = load_yolo_style_data("/content/data", "val")

    data = {"X_train": X_train, "y_train": y_train, "X_val": X_val, "y_val": y_val}

    if X_train.size > 0:
        print(f"Đã nạp {len(X_train)} ảnh train. Bắt đầu huấn luyện...")
        model_wrapper.train(data=data, epochs=25, batch_size=16)
        model_wrapper.save()


In [ ]:
from google.colab import files

files.download('models/face_verify_v1.keras')